# 記事の見出しを生成して実際の見出しと比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_summarization/headline_generation.ipynb)

NLP クックブック 23-08 ／ 要約と平易化

記事の本文から、内容を一言で表す見出し (タイトル) を作りたい。極端に短い要約として、生成結果を実際の見出しと比べたい。

In [ ]:
!pip install -q rouge-score fugashi unidic-lite

In [ ]:
import io, tarfile, urllib.request
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
df = df[df["label"].isin(["topic-news", "sports-watch"])].sample(12, random_state=1).reset_index(drop=True)

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def headline(text):
    prompt = ("次のニュース記事に、30 字以内の見出しを 1 つ付けてください。体言止めにし、記事にない情報は入れないでください。見出しだけを出力してください。\n\n" + text[:600])
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().split("\n")[0].strip("「」『』")

df["generated"] = [headline(t) for t in df["text"]]
df["lead"] = [t.strip().split("。")[0][:30] for t in df["text"]]      # ベースライン: 本文の最初の文の先頭 30 字
for _, r in df.head(5).iterrows():
    print(f"実際: {r['title']}\n生成: {r['generated']}\n先頭: {r['lead']}\n")

In [ ]:
import numpy as np
from fugashi import Tagger
from rouge_score import rouge_scorer

tagger = Tagger()

class Tok:
    def tokenize(self, text):
        return [w.surface for w in tagger(text)]

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], tokenizer=Tok())
for col, label in [("generated", "LLM の見出し"), ("lead", "本文の先頭 30 字")]:
    scores = [scorer.score(t, g) for t, g in zip(df["title"], df[col])]
    print(f"{label:12} ROUGE-1 F = {np.mean([s['rouge1'].fmeasure for s in scores]):.3f}  ROUGE-L F = {np.mean([s['rougeL'].fmeasure for s in scores]):.3f}  平均 {df[col].str.len().mean():.0f} 字")

- 見出し生成は、要約の中でも最も圧縮率が高いタスク。何を残すかの選択がそのまま品質になる。
- 実際の見出しは、読者の興味を引くための表現 (煽り、伏せ字、疑問形) を含むことが多く、内容の要約とは限らない。ROUGE が低くても、見出しとして悪いとは言えない。
- 12 記事では平均値のばらつきが大きい。手法を比べるなら、数百件以上で評価し、信頼区間を付ける (評価の章を参照)。
- 「記事にない情報は入れない」と指示しても、固有名詞や数字を作ってしまうことがある。公開する見出しに使うなら、人の確認が必要。